[tinyurl.com/upenn-history-workshop](https://tinyurl.com/upenn-history-workshop)

## 1. The International Image Interoperability Framework (IIIF) Manifest
### Find a IIIF manifest in Colenda

Libraries and museums publish their collections online so you can view scanned materials in a web browser. Behind the scenes, many institutions also make their collections available as **structured data** — meaning a computer can read and process them, not just display them as a web page.

They do this using an open standard called the **International Image Interoperability Framework (IIIF)**. A IIIF **manifest** is a special file (in JSON format) that contains links to all the images in a collection, along with descriptive metadata (title, author, date, etc.).

**To find a UPenn manifest:**

1. Go to Penn's digital repository: https://colenda.library.upenn.edu/
2. Browse or search for a collection that interests you.
3. Open an item and look for the **IIIF presentation manifest** link in the upper-right corner of the page.
4. Clicking that link will give you a URL — this is what we'll paste into the code below.

For this demo, we'll use a draft letter by Marian Anderson:
https://colenda.library.upenn.edu/catalog/81431-p3rx93h4f

Possibly related to this event? https://timesmachine.nytimes.com/timesmachine/1978/03/25/110813292.html?pageNumber=21

Full finding aid here: https://findingaids.library.upenn.edu/records/UPENN_RBML_PUSP.MS.COLL.200

You can also change the manifest to load any collection that you wish from the world's [libraries, mueseums and archives](https://www.library.universiteitleiden.nl/binaries/content/assets/ul2ub/bijzondere-collecties/list-of-iiif-collections.pdf).


## 2. How to use this notebook

This is a **Google Colab notebook** — an interactive document that mixes text (like what you're reading now) with runnable code. You don't need to install anything; it runs in your browser. It is shared with you and available to use after the workshop. You are also welcome to make your own copy (File > Save a Copy).

- **To run a code cell**, click the **play button** (▶) to its left, or press **Shift + Enter**.
- Run the cells **in order from top to bottom**. Each cell may depend on results from earlier cells.
- You don't need to understand the code to use this notebook — just follow the instructions and run each cell when prompted.

In [ ]:
manifest_url = "https://colenda.library.upenn.edu/items/ark:/81431/p3rx93h4f/manifest"
# Note that the manifest URL needs to be in quotes

import requests
import srsly
from rich.console import Console
from rich.text import Text

if not "https" in manifest_url:
  manifest = srsly.read_json("/content/manifest.json")

else:
  header = {
      "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/11"
  }
  manifest = requests.get(manifest_url, headers=header)
  if manifest.status_code != 200:
      raise Exception(f"Error downloading manifest: {manifest.status_code}")

  manifest = manifest.json()
context = manifest.get('@context')
if '3' in context: # http://iiif.io/api/presentation/3/context.json
  console = Console()
  text = Text(str(manifest['label']['en'][0]), style="bold")
  for item in manifest.get('metadata', []):
    text.append(f"\n{item['label']['en'][0]}:{item['value']['en'][0]}")

  console.print(text)

elif '2' in context:
  console = Console()
  text = Text(str(manifest['label']), style="bold")
  if manifest.get('description'):
    text.append("\n\n" + manifest['description'], style="italic")

  for item in manifest.get('metadata', []):
    text.append(f"\n{item['label']}:{item['value']}")

  console.print(text)

else:
  print('unsuported IIIF presentation version number')


## Extracting Image URLs from the Manifest

With IIIF, every image in a collection has its own unique web address (URL). The manifest file we loaded above contains all of those links. The code below reads the manifest and builds a list of image URLs so we can download and work with them.

**Just run the cell below** — it will display the list of image URLs it found.

In [ ]:
from tqdm import tqdm

context = manifest.get('@context')

if "2" in context:
  image_urls = []
  for sequence in manifest.get('sequences',[]):
    for canvas in sequence.get('canvases',[]):
      for rendering in canvas.get('rendering',[]):
        image_urls.append(rendering['@id'])

if "3" in context:
  image_urls = manifest.get('images', [])

image_urls


## Viewing the Images

Now let's download and display the actual images. In this case, each image is a scanned page from an archival document.

**Run the cell below** to see the images. Take a moment to look at them — what do you notice about the document?

In [ ]:
import os
import requests
from PIL import Image
from io import BytesIO
from IPython.display import display

# create iiif_image directory
if not os.path.exists('iiif_images'):
    os.makedirs('iiif_images')

# download images
for i, image_url in enumerate(image_urls):
  response = requests.get(image_url)
  if response.status_code == 200:
    img = Image.open(BytesIO(response.content))
    # convert to jpeg
    img = img.convert('RGB')
    img.save(f'iiif_images/{i}.jpg', 'JPEG')
    display(img)


# Working with Your Own Files

The steps above used images from an online IIIF manifest. But you may also want to work with materials you've gathered yourself — photographs, scanned documents, or images downloaded from the web.

In the next few cells, we'll connect to **Google Drive** so you can point the notebook at a folder of your own files. If you don't have files to use, you can skip ahead to the results section.

In [ ]:
# @title Click if you're also using files from Drive
DRIVE = False # @param {"type":"boolean","placeholder":"True"}


> **Running outside Google Colab?**  
> This notebook has been adapted for use in any Jupyter environment (local Jupyter, VS Code, JupyterHub, etc.).  
> Colab-specific cells (file upload, Google Drive mount, secret manager) have been replaced with portable equivalents.  
> See the comments in each affected cell for instructions.
>
> To use your own data, place files in a `data/` folder next to this notebook, or set the `DATA_FOLDER`
> environment variable to the path of your data directory.  
> Store API keys in a `.env` file (requires `pip install python-dotenv`) or set them as environment variables.


In [ ]:
# from google.colab import drive  # Colab only — replaced below
# drive.mount('/content/drive')  # Colab only
# Set this to your local data folder instead:
import os
drive_folder = os.environ.get('DATA_FOLDER', 'data/')

### Set the path to your folder

After mounting Google Drive above, you need to tell the notebook where your images are.

1. In the left sidebar of Colab, click the **folder icon** (📁) to open the file browser.
2. Navigate to your folder inside `drive/MyDrive/...`.
3. Right-click the folder and choose **"Copy path"**.
4. Paste that path into the code cell below, replacing the placeholder text (keep the quotes).

In [ ]:
folder_path = "/content/drive/MyDrive/ark-81431"

### Convert PDFs to Images

If your folder contains PDF files, the code below will convert each page into a JPEG image so the OCR model can read them. **Run both cells below** — the first installs the necessary software, and the second does the conversion.

*(If you don't have any PDFs, these cells will simply do nothing.)*

In [ ]:
%pip install -q pymupdf

In [ ]:
import pymupdf
from pathlib import Path
from tqdm import tqdm

# Find all PDFs
pdfs = list(Path(folder_path).glob("**/*.pdf"))
print(f"Found {len(pdfs)} PDF files")

# Convert to JPGs
for pdf_file in tqdm(pdfs, desc="Converting PDFs"):
    try:
        doc = pymupdf.open(pdf_file)
        for i, page in enumerate(doc):
            jpg_path = pdf_file.parent / f"{pdf_file.stem}_{i:03d}.jpg"
            if not jpg_path.exists():
                pix = page.get_pixmap(dpi=150)
                img = pix.pil_image()
                img.save(jpg_path)
        doc.close()
    except Exception as e:
        print(f"⚠️  Error processing {pdf_file.name}: {e}")

# Count all images
all_images = list(Path(folder_path).glob("**/*.jpg")) + list(Path(folder_path).glob("**/*.png"))

print(f"\n📸 Total images found: {len(all_images)}")

### Convert HEIC Images (iPhone Photos)

Photos taken on iPhones are often saved in HEIC format. The code below converts any HEIC files in your folder to standard JPEG images.

**Run both cells below.** *(If you don't have any HEIC files, these cells will simply do nothing.)*

In [ ]:
%pip install -q pillow-heif

In [ ]:
from PIL import Image
from pathlib import Path
from tqdm.notebook import tqdm
from pillow_heif import register_heif_opener

# Register the HEIF opener
register_heif_opener()

drive_heic = Path(folder_path).glob('**/*.HEIC')

for heic_file in tqdm(drive_heic):
  img = Image.open(heic_file)
  img.save(f"{heic_file.parent}/{heic_file.stem}.jpg")


# Setting Up the AI Model (API Key)

To extract text from images, we'll send them to an AI vision model hosted by Alibaba Cloud (called **Qwen**). This model is open-source, meaning you can also run it on your own computer or university servers — but for today, we'll use Alibaba's hosted version, which requires an **API key** (like a password that lets you connect).

### For this workshop:
We'll provide a key you can use during the session. **It will stop working after the workshop ends.**

### To add the API key in Colab:
1. Click the **key icon** (🔑) in the left sidebar.
2. Click **"Add new secret"**.
3. Under **Name**, enter: `DASHSCOPE_API_KEY`
4. Under **Value**, paste the key we provide: `sk-90713447bd5648d7939e1e51857307b4`

### To get your own key after the workshop:
- [Create an Alibaba Cloud account](https://account.alibabacloud.com/register/intl_register.htm)
- [Create an API key](https://www.alibabacloud.com/help/en/model-studio/get-api-key)
- You can also use other providers — see the [full list here](https://openrouter.ai/qwen/qwen3-vl-8b-instruct/providers).

In [ ]:
# @title Image Encoding Function
import base64
from pathlib import Path

def encode_images(image_directory: str, file_extensions: list = ['.jpg', '.png'], data_uri: bool = True):
    """Lazy generator that encodes images on-demand.

    Args:
        image_directory: Path to directory containing images
        file_extensions: List of file extensions to process
        data_uri: Include data URI scheme in output

    Yields:
        tuple: (Path, base64_string) for each image
    """
    image_directory = Path(image_directory)
    if not image_directory.is_dir():
        raise ValueError(f"{image_directory} is not a valid directory")

    for path in image_directory.rglob('*'):
        if path.is_file() and path.suffix.lower() in file_extensions:
            with open(path, 'rb') as img_file:
                encoded = base64.b64encode(img_file.read()).decode('utf-8')
                if data_uri:
                    mime = 'image/jpeg' if path.suffix.lower() == '.jpg' else 'image/png'
                    yield path, f"data:{mime};base64,{encoded}"
                else:
                    yield path, encoded

In [ ]:
# from google.colab import userdata  # Colab only — replaced below
import os
# Load API keys from environment variables or a .env file:
# pip install python-dotenv  then add your key to a .env file
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass  # install python-dotenv if you use a .env file

base_url="https://dashscope-intl.aliyuncs.com/compatible-mode/v1"
api_key= os.environ.get('DASHSCOPE_API_KEY') # We will provide you with an API key for the workshop
model = "qwen3-vl-plus" #or "qwen3-vl-flash" (faster and cheaper), "qwen-vl-ocr" (for typewritten texts) For a full list of available models, see https://www.alibabacloud.com/help/en/model-studio/models

In [ ]:
prompt = "Extract all text from the image. Be sure to include any text that is crossed-out and format ~~Crossed out.~~. Return text as markdown"

In [ ]:
# @title Code that sends the encoded image to the server
import os
import asyncio
import aiohttp
from openai import AsyncOpenAI
from tqdm.asyncio import tqdm

# Use AsyncOpenAI instead of OpenAI
client = AsyncOpenAI(
    api_key=api_key,
    base_url=base_url,
)

async def transcribe_image(img: tuple, session: aiohttp.ClientSession, semaphore: asyncio.Semaphore):
    """Process a single image with rate limiting and timeout"""
    async with semaphore:  # Limit concurrent requests
        image_path = img[0]
        image_b64 = img[1]
        file = {
            "name": image_path,
        }
        #print(f"Starting to process: {image_path.name}")

        try:
            # Add timeout to API call as well
            completion = await asyncio.wait_for(
                client.chat.completions.create(
                    model=model,
                    messages=[
                        {
                            "role": "user",
                            "content": [
                                {
                                    "type": "text",
                                    "text": prompt
                                },
                                {
                                    "type": "image_url",
                                    "image_url": {
                                        "url": image_b64
                                    }
                                }
                            ]
                        }
                    ],
                ),
                timeout=360  # 360 second timeout for API call
            )
            #print(f"Successfully processed: {image_path.name}")
            file["markdown_text"] = completion.choices[0].message.content
            return file
        except asyncio.TimeoutError:
            print(f"API timeout for image {image_path.name}")
            return None
        except Exception as e:
            print(f"Error processing image {image_path.name}: {e}")
            return None

async def transcribe_all_images(images, max_concurrent=3):
    """Process all images concurrently with a limit on concurrent requests"""
    # Create aiohttp session for connection pooling
    connector = aiohttp.TCPConnector(limit=10, limit_per_host=5)
    timeout = aiohttp.ClientTimeout(total=300)  # 5 minute total timeout

    async with aiohttp.ClientSession(connector=connector, timeout=timeout) as session:
        semaphore = asyncio.Semaphore(max_concurrent)

        # Create tasks for all images
        tasks = [transcribe_image(image, session, semaphore) for image in images]

        # Process with progress bar
        results = []
        completed_count = 0

        for task in tqdm.as_completed(tasks, total=len(tasks), desc="Processing images"):
            try:
                result = await task
                completed_count += 1
                if result is not None:
                    results.append(result)
                #print(f"Completed {completed_count}/{len(tasks)} images")
            except Exception as e:
                print(f"Task failed with error: {e}")
                completed_count += 1

    return results




In [ ]:
# needed to run async in Jupyter
import nest_asyncio
nest_asyncio.apply()

In [ ]:
import itertools

# Step 1: Pool images from both IIIF and Drive folder
iiif_images = encode_images("/content/iiif_images")  # Images from manifest
if DRIVE:
  drive_images = encode_images(folder_path)  # Images from local folder
  all_images = list(itertools.chain(iiif_images, drive_images))
else:
  all_images = list(iiif_images)
print(f"Encoded {len(all_images)} images")


print("🚀 Starting OCR pipeline...")

# Step 2: Process all images
results = asyncio.run(transcribe_all_images(all_images, max_concurrent=15))

print(f"✅ Processed {len(results)} images successfully")

## Cleaning Up the Raw Results

The AI model returns its response wrapped in formatting tags (like ` ```markdown ``` `) because it's designed for chatbot interfaces. The code below strips those tags and cleans up repeated text so we're left with just the extracted content.

**Run the next few cells** to view the raw results and then clean them.

In [ ]:
results

In [ ]:
# @title Clean text helper functions
import re

def remove_code_tags(text):
    """Remove markdown code fences."""
    return re.sub(r"```[a-zA-Z]*\n(.*?)\n```", r"\1", text, flags=re.DOTALL)

def remove_repeated_phrases(text):
    """Remove consecutive repeated phrases."""
    pattern = r"(\b\w+(?: \w+){0,5}\b)( \1)+"
    return re.sub(pattern, r"\1", text)

def clean_text(text):
    """Clean extracted text."""
    text = remove_code_tags(text)
    text = remove_repeated_phrases(text)
    return text.strip()

In [ ]:
# Step 3: Clean the text
for result in tqdm(results, desc="Cleaning text"):
    result["text"] = clean_text(result["markdown_text"])


## Comparing Images with Extracted Text

Now let's see the original images side-by-side with the text the AI extracted. This is a good way to check the quality of the transcription and spot any errors.

**Run the cell below** to display each image alongside its transcription.

In [ ]:
from PIL import Image, ImageDraw
from IPython.display import display, Markdown

# Display image from a local file path
for result in results:
    img = Image.open(result['name'])
    img = img.resize((500, 500))
    display(img)
    display(Markdown(result['text']))





## Saving Results as a Spreadsheet

You can export the extracted text to a **CSV file** (a spreadsheet format that opens in Excel, Google Sheets, etc.). Each row will represent one page/image.

**Run the cell below** to create the spreadsheet.

In [ ]:
import pandas as pd
# Save as CSV
df = pd.DataFrame(results)
# drop raw text
df = df.drop(columns=["markdown_text"])
# sort by name
df = df.sort_values(by=['name'])
output_csv = "/content/" + "ocr_results.csv"
df.to_csv(output_csv, index=False)
df

### Download the Spreadsheet

**Run the cell below** to download the CSV file to your computer. Your browser will prompt you to save it.

In [ ]:
from google.colab import files
files.download('/content/ocr_results.csv')

# Building a Simple Searchable Website

Another way to share your work is to create a small **static website** with full-text search. The code below generates an HTML page for each image and its transcription, plus a search page powered by [Pagefind](https://pagefind.app/).

**Run the next several cells in order.** They will:
1. Create an HTML page for each image.
2. Create a search index page.
3. Install and run the Pagefind search indexer.
4. Package everything into a ZIP file for download.

In [ ]:
import os
import markdown
from pathlib import Path

if not Path('_site').exists():
  Path('_site').mkdir()

# copy image to _site and update data
for image in results:
  Path(f"_site/{image['name'].name}").write_bytes(Path(image['name']).read_bytes())
  image['name'] = image['name'].name
images = sorted(results, key=lambda x: x.get('name',''))
first_image = images[0]['name'] + ".html"
last_image = images[-1]['name'] + ".html"
for idx, image in enumerate(results):
    next_image = images[idx+1]['name'] + ".html" if idx < len(images)-1 else first_image
    prev_image = images[idx-1]['name'] + ".html" if idx > 0 else last_image
    cleaned_text = image.get('text','')
    cleaned_text = markdown.markdown(cleaned_text, extensions=['tables'])
    sort_value = int("".join([char for char in image["name"] if char.isdigit()]))
    html_template = f"""
    <!DOCTYPE html>
    <html>
      <body>
        <button><a href="/">Back to Search</a></button>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;
        <button><a href="{prev_image}">Back</a></button>
        <button><a href="{next_image}">Forward</a></button>

        <h1 data-pagefind-sort="page:{sort_value}">{image["name"]}</h1>
        <img src="{image["name"]}">
        <p>{cleaned_text}</p>

      </body>
    </html>
    """
    Path(f"_site/{image['name']}.html").write_text(html_template)

In [ ]:
index = f"""

<!DOCTYPE html>
        <html>
          <head>
          <link href="/pagefind/pagefind-ui.css" rel="stylesheet">
          <script src="/pagefind/pagefind-ui.js"></script>
          </head>
          <body>
            <div id="search"></div>
            <script>
                window.addEventListener('DOMContentLoaded', (event) => {{
                    new PagefindUI({{
                      element: "#search",
                      sort: {{ date: "desc" }},
                      showSubResults: true
                    }});

                }});
            </script>
             </body>
        </html>
"""
Path('_site/index.html').write_text(index)

In [ ]:
%pip install -q 'pagefind[extended]'

In [ ]:
!python3 -m pagefind --site _site

In [ ]:
import shutil
zip_file = shutil.make_archive('site', 'zip', '/content/_site')

In [ ]:
from google.colab import files
files.download(zip_file)

Here's a demo of a really simple website to search, read and share archival materials

https://heartfelt-twilight-6cc7f5.netlify.app/

### Viewing Your Website on your Computer

After downloading and unzipping the file on your computer:
1. Open the unzipped folder.
2. Double-click **index.html** to open it in your web browser.
3. Use the search bar to find text across all your transcribed pages.

> **Note:** The search feature only works if you open the site through a local web server. If search doesn't work by double-clicking, try opening a terminal and running `python3 -m http.server` from inside the unzipped folder, then visit `http://localhost:8000` in your browser.

# Going Further: A More Polished Website with Flatfish

[Flatfish](https://github.com/apjanco/flatfish) is a tool that builds a richer, more polished website from your images and transcriptions — with better navigation, search, and layout. The cells below show how to set up and deploy a Flatfish project.

**Run each cell in order.** You will need your API key set up (see above).

In [ ]:
%pip install flatfish -q

In [ ]:
from pathlib import Path

# copy all jpg files to a new train directory
images = Path("/content/iiif_images").glob("*.jpg")

if not Path("/content/train").exists():
  Path("/content/train").mkdir()

for image in images:
  image.rename("/content/train/" + image.name)

In [ ]:
dataset_id = "apjanco/demo_images"

In [ ]:
from datasets import load_dataset
dataset = load_dataset("imagefolder", data_dir="/content/")
dataset.push_to_hub(dataset_id)

In [ ]:
!flatfish init marion-anderson

In [ ]:
import os
# from google.colab import userdata  # Colab only — replaced below
import os
# Load API keys from environment variables or a .env file:
# pip install python-dotenv  then add your key to a .env file
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass  # install python-dotenv if you use a .env file

# For step #2 we'll use our key saved in secrets
os.environ["DASHSCOPE_API_KEY"] = os.environ.get('DASHSCOPE_API_KEY')

In [ ]:
%cd marion-anderson

In [ ]:
!flatfish validate

Be sure to check that your dataset_id matches what you see in Dataset: above

In [ ]:
!flatfish process

In [ ]:
import os
# from google.colab import userdata  # Colab only — replaced below
import os
# Load API keys from environment variables or a .env file:
# pip install python-dotenv  then add your key to a .env file
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass  # install python-dotenv if you use a .env file

# For step #2 we'll use our key saved in secrets
os.environ["NETLIFY_TOKEN"] = os.environ.get('NETLIFY_TOKEN')

In [ ]:
!flatfish deploy --site 2cde00d3-c515-4e1b-ad5e-e898c0f9d26e